<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [13]</a>'.</span>

# PRewrite

PRewrite (Kong et al., 2024) is an input control method that rewrites a seed instruction into a more effective form using an LLM as a rewriter. Three strategies from the original paper are supported:

  - **PRewrite-I** (`strategy="inference"`): generate a single greedy rewrite. This is relatively cheap; useful when a quick instruction rewrite is enough.
  - **PRewrite-S** (`strategy="search"`): sample `k_candidates` rewrites and select the best by evaluating each on a held-out dev set. This is more expensive but is generally of higher quality.
  - **RL-trained rewriter** (by setting `train_rewriter=True`): train the rewriter with GRPO (group-relative policy optimization, via the TRL wrapper layer at `structural_control/wrappers/trl/`) before proposing rewrites; the trained rewriter is then consumed by either PRewrite-I or PRewrite-S. The reward is the downstream task metric (apply each rewrite with the frozen task model over a dev set and score it; this is the paper's metric-in-the-loop reward), or a user-supplied `reward_fn`. The toolkit's implementation of PRewrite uses GRPO rather than the paper's PPO since GRPO takes a callable reward (so the reward can be the task metric directly).

PRewrite's chosen rewrite is applied as the system prompt at inference time.

Reference: [Kong et al., 2024; PRewrite: Prompt Rewriting with Reinforcement Learning](https://arxiv.org/abs/2401.08189).

## Method parameters

| parameter                     | type                  | description                                                                                                |
| ----------------------------- | --------------------- | ---------------------------------------------------------------------------------------------------------- |
| `initial_instruction`         | `str`                 | Seed instruction to rewrite. Required.                                                                     |
| `rewriter_model_name_or_path` | `str \| None`         | HF id or local path for the rewriter LLM. `None` reuses the task model as rewriter (forbidden under training). |
| `rewriter_model`              | `PreTrainedModel \| None` | Pre-loaded rewriter instance; mutually exclusive with `rewriter_model_name_or_path`.                    |
| `rewriter_tokenizer`          | `PreTrainedTokenizer \| None` | Tokenizer paired with `rewriter_model` (required if the model has no inferable name_or_path).      |
| `meta_prompt`                 | `str \| None`         | Custom rewriter meta-prompt template (must contain `{seed}`). `None` uses the toolkit default.             |
| `strategy`                    | `"inference" \| "search"` | PRewrite-I (greedy single rewrite) or PRewrite-S (best-of-k by dev-set eval).                          |
| `k_candidates`                | `int`                 | Number of candidate rewrites to sample under the search strategy.                                          |
| `dev_set`                     | `list[dict] \| None`  | Held-out rows used to score candidates (search strategy) and as the GRPO metric-in-the-loop reward (training). Each row needs `"input"` and optionally `"reference"`. |
| `metric`                      | `Metric \| None`      | An aisteer360 `Metric` aggregating dev-set responses into a scalar (used by the search strategy and the GRPO reward). |
| `score_key`                   | `str \| None`         | When the metric returns a dict, which key to extract. Falls back to the first numeric value.               |
| `train_rewriter`              | `bool`                | If `True`, GRPO-train the rewriter before proposing rewrites. Requires an explicit rewriter and a reward source (`reward_fn`, or `metric` + `dev_set`). |
| `reward_fn`                   | `Callable \| None`    | Custom GRPO reward `reward_func(prompts, completions, **kwargs) -> list[float]`. Takes precedence over `metric` + `dev_set`; if unset, the reward is built from `metric` + `dev_set` (the paper's reward). |
| `training_seeds`              | `list[str] \| None`   | Pool of seed instructions used as GRPO rollout prompts. Defaults to `[initial_instruction]`.               |
| `grpo_config`                 | `dict \| None`        | Configuration forwarded to TRL's GRPO trainer (see `GRPOArgs` for keys: `num_generations`, `beta`, `max_completion_length`, `learning_rate`, `per_device_train_batch_size`, ...). |
| `reward_dev_size`             | `int \| None`         | Cap on dev rows used per reward evaluation during GRPO training (cost control); a deterministic head slice of `dev_set`. |
| `rewriter_gen_kwargs`         | `dict \| None`        | Generation kwargs for the rewriter LLM (e.g. temperature, max_new_tokens).                                 |
| `eval_gen_kwargs`             | `dict \| None`        | Generation kwargs used when scoring candidates against the dev set (search selection and the GRPO reward). |


## Setup

If running this from a Google Colab notebook, please uncomment the following cell to install the toolkit. The following block is not necessary if running this notebook from a virtual environment where the toolkit has already been installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub using a `HUGGINGFACE_TOKEN` value stored in a `.env` file.

In [2]:
# !pip install python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: Rewriting a vague instruction for short-answer extraction

We start with the seed instruction `"Please provide an answer to the question."` and ask PRewrite to produce a more specific instruction that nudges the model toward concise factual answers. We compare the greedy PRewrite-I rewrite, PRewrite-S which evaluates `k_candidates` rewrites on a small dev set of factual Q/A pairs and keeps the one with the highest F1, and the GRPO-trained rewriter.

In [3]:
import gc
import os
import warnings

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from aisteer360.algorithms.input_control.prewrite import PRewrite
from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.evaluation.metrics.generic.short_answer_match import ShortAnswerMatch

warnings.filterwarnings('ignore', category=UserWarning)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

SEED_INSTRUCTION = "Please provide an answer to the question."

/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Define a small dev set of short-answer factual Q/A pairs and a held-out test prompt:

In [4]:
dev_set = [
    {"input": "What's the capital of France?", "reference": "Paris"},
    {"input": "What's the capital of Australia?", "reference": "Canberra"},
    {"input": "How many planets are in the Solar System?", "reference": "8"},
    {"input": "How many sides does a hexagon have?", "reference": "6"},
    {"input": "What's the chemical symbol for sodium?", "reference": "Na"},
    {"input": "What's the chemical symbol for potassium?", "reference": "K"},
    {"input": "Who painted the Mona Lisa?", "reference": "Leonardo da Vinci"},
    {"input": "Who wrote 'Pride and Prejudice'?", "reference": "Jane Austen"},
    {"input": "What year did World War II end?", "reference": "1945"},
    {"input": "What year did the Berlin Wall fall?", "reference": "1989"},
    {"input": "What's the largest ocean?", "reference": "Pacific"},
    {"input": "What's the smallest planet?", "reference": "Mercury"},
    {"input": "Which planet is known as the Red Planet?", "reference": "Mars"},
    {"input": "What's the hardest natural substance?", "reference": "diamond"},
    {"input": "What's the freezing point of water in Celsius?", "reference": "0"},
    {"input": "What language has the most native speakers?", "reference": "Mandarin"},
    {"input": "What's the currency of Japan?", "reference": "yen"},
    {"input": "How many continents are there?", "reference": "7"},
    {"input": "What's the speed of light in km/s (approx)?", "reference": "300000"},
    {"input": "Who discovered penicillin?", "reference": "Alexander Fleming"},
    {"input": "What's the longest river in the world?", "reference": "Nile"},
    {"input": "What's the chemical formula for water?", "reference": "H2O"},
]

PROMPT = "Who wrote the novel '1984'?"  # held-out; not in dev_set

### Evaluation metric

We score short-answer responses with the toolkit's `ShortAnswerMatch` metric (`aisteer360.evaluation.metrics`), which implements the standard SQuAD exact-match and token-level F1 (Rajpurkar et al., 2016). We select rewrites on **F1** (`score_key="f1"`): its precision term penalizes verbose answers that merely contain the gold span, so it rewards concise, correct answers without saturating. `TaskEvaluationScorer` passes the gold answers as both `references` and `reference_answers`; the metric accepts either.

As a quick sanity check, we run the metric over the `dev_set` against a small set of illustrative responses.

In [5]:
_metric = ShortAnswerMatch()
for _resp, _ref in [
    ("Paris", "Paris"),  # concise + correct
    ("The capital of France is Paris.", "Paris"),  # correct but verbose -> lower F1
    ("London", "Paris"),  # wrong -> 0
]:
    _out = _metric.compute(responses=[_resp], references=[_ref])
    print(f"EM={_out['exact_match']:.0f}  F1={_out['f1']:.2f}  ref={_ref!r:9}  resp={_resp!r}")


EM=1  F1=1.00  ref='Paris'    resp='Paris'
EM=0  F1=0.33  ref='Paris'    resp='The capital of France is Paris.'
EM=0  F1=0.00  ref='Paris'    resp='London'


### Baseline model behavior

We start by generating a response with the bare seed instruction (no rewriting), to provide a reference for the comparisons below.

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

baseline_chat = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SEED_INSTRUCTION},
        {"role": "user", "content": PROMPT},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
baseline_inputs = tokenizer(baseline_chat, return_tensors="pt").to(model.device)
baseline_output = model.generate(**baseline_inputs, max_new_tokens=64, do_sample=False)
baseline_response = tokenizer.decode(
    baseline_output[0, baseline_inputs.input_ids.size(1):],
    skip_special_tokens=True,
)
print("Baseline response:\n")
print(baseline_response)

del model, baseline_inputs, baseline_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|██▌       | 1/4 [00:08<00:25,  8.57s/it]

Loading checkpoint shards:  50%|█████     | 2/4 [00:17<00:17,  8.55s/it]

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:25<00:08,  8.47s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:27<00:00,  5.98s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:27<00:00,  6.91s/it]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline response:

The novel '1984' was written by George Orwell.


### Aligning the rewriter with the metric

By default, PRewrite's rewriter optimizes for general clarity and thoroughness, which tends to make answers longer. This demo aims to show how to rewrite instructions to induce more concise answers and, since the metric only selects among the candidates the rewriter proposes, we define a custom `meta_prompt` that asks for instructions eliciting short, exact answers. Note that the template must contain `{seed}`.

In [7]:
CONCISE_META_PROMPT = (
    "You are a prompt engineer. Rewrite the instruction below into an instruction telling the model "
    "to reply with ONLY the answer, e.g., the number, name, place, etc.\n\n"
    "Original instruction:\n{seed}\n\n"
    "Rewritten instruction:"
)

### PRewrite-I (greedy single rewrite)

The simplest mode is to ask the rewriter LLM (in this case we use the same model for both the task and rewriter) to produce one improved instruction (under greedy-decoding). This is cheap to run and does not use the dev set.

In [8]:
prewrite_i = PRewrite(
    initial_instruction=SEED_INSTRUCTION,
    meta_prompt=CONCISE_META_PROMPT,
    strategy="inference",
    rewriter_gen_kwargs={"max_new_tokens": 64, "do_sample": False},
)
pipeline_i = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[prewrite_i],
    device_map="auto",
    hf_model_kwargs={"torch_dtype": torch.bfloat16},
)
pipeline_i.steer()

print("Chosen rewrite (PRewrite-I):\n")
print(prewrite_i.memory["instruction"])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|██▌       | 1/4 [00:02<00:08,  2.71s/it]

Loading checkpoint shards:  50%|█████     | 2/4 [00:05<00:05,  2.69s/it]

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:07<00:02,  2.65s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  1.85s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.15s/it]

Chosen rewrite (PRewrite-I):

Provide the answer.


In [ ]:
response_i = pipeline_i.generate(
    messages=[{"role": "user", "content": PROMPT}],
    max_new_tokens=64,
    do_sample=False,
)
print("Response (PRewrite-I):\n")
print(response_i)

del pipeline_i
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### PRewrite-S (best-of-k by dev-set evaluation)

The `PRewrite-S` strategy samples `k_candidates` rewrites with `do_sample=True`, then runs each on the dev set under our `ShortAnswerMatch` metric and keeps the rewrite with the highest mean accuracy.

We use a small `k_candidates=3` for illustration purposes. The dev rows for each candidate are evaluated in a single batched generate call by `TaskEvaluationScorer`.

In [10]:
prewrite_s = PRewrite(
    initial_instruction=SEED_INSTRUCTION,
    meta_prompt=CONCISE_META_PROMPT,
    strategy="search",
    k_candidates=3,
    dev_set=dev_set,
    metric=ShortAnswerMatch(),
    score_key="f1",
    rewriter_gen_kwargs={"max_new_tokens": 160, "do_sample": True, "temperature": 0.7},
    eval_gen_kwargs={"max_new_tokens": 16, "do_sample": False},
)
pipeline_s = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[prewrite_s],
    device_map="auto",
    hf_model_kwargs={"torch_dtype": torch.bfloat16},
)
pipeline_s.steer()

print("Chosen rewrite (PRewrite-S):\n")
print(prewrite_s.memory["instruction"])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|██▌       | 1/4 [00:02<00:08,  2.70s/it]

Loading checkpoint shards:  50%|█████     | 2/4 [00:05<00:05,  2.68s/it]

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:07<00:02,  2.64s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  1.89s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.17s/it]

Chosen rewrite (PRewrite-S):

Provide the answer only.


In [ ]:
response_s = pipeline_s.generate(
    messages=[{"role": "user", "content": PROMPT}],
    max_new_tokens=64,
    do_sample=False,
)
print("Response (PRewrite-S):\n")
print(response_s)

del pipeline_s
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Comparing the rewrites

The instructions under the seed vs. PRewrite-I's greedy rewrite vs. PRewrite-S's selected best-of-k are as follows:

In [12]:
print(f"Seed:           {SEED_INSTRUCTION}")
print("PRewrite-I:    ", prewrite_i.memory["instruction"])
print("PRewrite-S:    ", prewrite_s.memory["instruction"])

Seed:           Please provide an answer to the question.
PRewrite-I:     Provide the answer.
PRewrite-S:     Provide the answer only.


## GRPO-trained rewriter (`train_rewriter=True`)

PRewrite can optimize the rewriter directly against task performance by setting `train_rewriter=True`, which trains it with GRPO (group-relative policy optimization) via the TRL wrapper. We deviate from the original paper (which uses PPO) primarily since GRPO is critic-free (there is no separate reward model and no value model), just a callable reward. Here the reward is the paper's metric-in-the-loop reward, which PRewrite builds automatically from the `metric` + `dev_set` we already used for PRewrite-S. Each candidate rewrite is applied with the frozen task model over the dev set and scored with `ShortAnswerMatch` (F1). In other words, the same signal PRewrite-S uses to select rewrites now serves as the training reward. (To use a different reward, pass a callable `reward_fn(prompts, completions, **kwargs) -> list[float]`; it takes precedence over `metric` + `dev_set`.)

PRewrite forbids the default task-LM-as-rewriter behavior under training (training would mutate the task model in place, breaking downstream uses). Pass an explicit `rewriter_model_name_or_path` (or `rewriter_model=...` for a pre-loaded instance). The example below uses a smaller, explicit rewriter and keeps the task model (`MODEL_NAME`) as the frozen scorer for the reward, so the rewriter is optimized for the model the instruction is actually deployed on.

GRPO is configured via `grpo_config` (forwarded to TRL's `GRPOArgs`): `num_generations` is the group size G used for the group-relative advantage (must be >= 2 and evenly divide `per_device_train_batch_size`), `beta` is the KL-to-reference coefficient, and `max_completion_length` bounds the rewrite length. The metric reward is the expensive part (a full dev-set pass with the task model for every distinct rewrite in a group, every step), so we cap it with `reward_dev_size` and keep the dev set, group size, and epoch count small for this illustration.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [13]:
REWRITER_NAME = "meta-llama/Llama-3.2-3B-Instruct"  # smaller than the task LM; the trainable rewriter

training_seeds = [
    "Answer the question.",
    "Reply concisely.",
    "Provide the answer.",
    "Respond with just the answer.",
    "Give a short, direct answer.",
    "Answer briefly and precisely.",
    "State the answer only.",
    "Return the exact answer.",
]

prewrite_grpo = PRewrite(
    initial_instruction=SEED_INSTRUCTION,
    meta_prompt=CONCISE_META_PROMPT,
    strategy="inference",
    rewriter_model_name_or_path=REWRITER_NAME,  # explicit rewriter; the task LM is never trained
    train_rewriter=True,
    metric=ShortAnswerMatch(),  # metric-in-the-loop reward
    dev_set=dev_set,
    score_key="f1",
    reward_dev_size=8,  # cap dev rows per reward eval (cost control); the reward runs the 8B task LM
    training_seeds=training_seeds,
    grpo_config={
        "num_train_epochs": 2,
        "num_generations": 4,  # GRPO group size G (>= 2)
        "per_device_train_batch_size": 4,  # must be divisible by num_generations
        "learning_rate": 1e-5,  # LoRA tolerates a higher LR than full-FT GRPO (TRL default 1e-6)
        "beta": 0.04,  # KL-to-reference coefficient
        "max_completion_length": 48,  # rewrites are short
        "use_peft": True,  # LoRA-by-default
    },
    rewriter_gen_kwargs={"max_new_tokens": 64, "do_sample": True, "temperature": 0.9},
    eval_gen_kwargs={"max_new_tokens": 16, "do_sample": False},  # used by the reward's task-LM passes
)

pipeline_grpo = SteeringPipeline(
    model_name_or_path=MODEL_NAME,  # frozen 8B task model: scores rewrites for the reward, then answers at inference
    controls=[prewrite_grpo],
    device_map="auto",
    hf_model_kwargs={"torch_dtype": torch.bfloat16},
)
pipeline_grpo.steer()  # GRPO trains the rewriter against the task metric, then proposes a single rewrite

print("Chosen rewrite (GRPO-trained rewriter, PRewrite-I):\n")
print(prewrite_grpo.memory["instruction"])

del pipeline_grpo
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|██▌       | 1/4 [00:02<00:08,  2.77s/it]

Loading checkpoint shards:  50%|█████     | 2/4 [00:05<00:05,  2.73s/it]

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:08<00:02,  2.69s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  1.88s/it]

...

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map: 100%|██████████| 8/8 [00:00<00:00, 2118.74 examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


TypeError: GRPOTrainer._get_train_sampler() takes 1 positional argument but 2 were given

## Quantifying instruction quality

PRewrite treats a rewrite as *input-agnostic* (one fixed instruction for all inputs) and judges it by the metric value when the model uses that instruction (accuracy for classification/reasoning, exact match for short-answer QA). In other words, the metric is the reward the rewriter optimizes.

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)  # show full instructions / generations

# held-out questions
test_set = [
    {"input": "What is the capital of Japan?", "reference": "Tokyo"},
    {"input": "Who developed the theory of general relativity?", "reference": "Einstein"},
    {"input": "What is the largest planet in the Solar System?", "reference": "Jupiter"},
    {"input": "In what year did the first crewed Moon landing occur?", "reference": "1969"},
    {"input": "Who wrote 'Romeo and Juliet'?", "reference": "Shakespeare"},
    {"input": "Which country gifted the Statue of Liberty to the United States?", "reference": "France"},
    {"input": "What is the tallest mountain on Earth?", "reference": "Everest"},
    {"input": "What gas do plants primarily absorb during photosynthesis?", "reference": "carbon dioxide"},
]

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model.generation_config.pad_token_id = tokenizer.eos_token_id  # silence the pad_token_id notice

The four instructions we compare are the seed and the three rewrites chosen above. For each one we have the task LM answer every held-out question.

In [ ]:
candidates = {
    "Seed":       SEED_INSTRUCTION,
    "PRewrite-I": prewrite_i.memory["instruction"],
    "PRewrite-S": prewrite_s.memory["instruction"],
    "GRPO":       prewrite_grpo.memory["instruction"],
}
references = [ex["reference"] for ex in test_set]

answers_by_method = {name: [] for name in candidates}
with torch.no_grad():
    for name, instruction in candidates.items():
        for ex in test_set:
            chat = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": instruction},
                    {"role": "user", "content": ex["input"]},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            enc = tokenizer(chat, return_tensors="pt").to(model.device)
            out = model.generate(**enc, max_new_tokens=64, do_sample=False)
            resp = tokenizer.decode(out[0, enc.input_ids.size(1):], skip_special_tokens=True)
            answers_by_method[name].append(resp.strip().replace("\n", " "))

We score each method with `ShortAnswerMatch` over the held-out set and report one row per method, alongside the instruction it used.

In [ ]:
metric = ShortAnswerMatch()
results = {name: metric.compute(responses=answers_by_method[name], references=references)
           for name in candidates}

# summary: one row per method (F1 + exact match + the instruction it used)
summary = pd.DataFrame({
    "method": list(candidates),
    "f1": [results[name]["f1"] for name in candidates],
    "exact_match": [results[name]["exact_match"] for name in candidates],
    "instruction": [candidates[name] for name in candidates],
})
display(summary)

Generations for each question and method are as follows.

In [ ]:
detail = pd.DataFrame({"question": [ex["input"] for ex in test_set], "reference": references})
for name in candidates:
    detail[name] = answers_by_method[name]
display(detail)

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Summary

This notebook demonstrated PRewrite (Kong et al., 2024), an input control that rewrites a seed instruction into a more effective form using an LLM as a rewriter, then applies the chosen rewrite as the system prompt at inference time. There are three variants of the method:

1. PRewrite-I (`strategy="inference"`) generates a single greedy rewrite. This strategy only costs one extra rewriter call before steering and it is appropriate for minor instruction edits.
2. PRewrite-S (`strategy="search"`) samples `k_candidates` rewrites and keeps the one with the highest mean score on a held-out `dev_set` under a `metric`. It costs `k × (rewriter call) + k × (dev_set size × task call)`; this method is useful when a dev set that captures the target behavior (and a good metric to score it) is available.
3. Setting `train_rewriter=True` runs GRPO (group-relative policy optimization, via the TRL wrapper) to train the rewriter on a pool of `training_seeds` before proposing rewrites; the trained rewriter is then consumed by either PRewrite-I or PRewrite-S. The GRPO reward is the same metric-in-the-loop signal PRewrite-S uses to *select* rewrites (apply each rewrite with the frozen task model over the `dev_set` and score it), now used as the *training* reward, or a user-supplied `reward_fn`. This optimizes the rewriter directly against task performance, at the cost of more compute (each reward evaluation runs the task model over the dev set for every distinct rewrite in a GRPO group).